| # | Topic |
|---|-------|
| 1 | Why Vector RAG Fails on Professional Documents |
| 2 | How PageIndex Builds a Tree Index from a PDF |
| 3 | LLM Tree Search — Reasoning Over Structure |
| 4 | Full End-to-End Vectorless RAG Pipeline |
| 5 | Expert-Guided Retrieval (Domain Knowledge Injection) |
| 6 | Chat API — Zero LLM Setup |
| 7 | Self-Hosted Open-Source Option |



## 🔑 Key Concept

> **Traditional RAG** → Chunk → Embed → Cosine Similarity → Retrieve
>
> **PageIndex RAG** → Build Tree → LLM Reasons Over Tree → Retrieve Exact Sections

### The Problem with Vector RAG

**Similarity ≠ Relevance**

A chunk about "market conditions" may score higher than the actual answer section simply because it shares more words with your query.


In [5]:
import os, json, time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")


load_dotenv()

llm = ChatOpenAI(
    model="prism-ml/ternary-bonsai-2-27b",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=1000,
)


PageIndex key loaded: ✅


In [6]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)


print("✅ PageIndex client ready")

✅ PageIndex client ready



## 🌲 Section 2: Upload & Index a PDF

### What Happens Here?

1. Upload your PDF to the PageIndex cloud.
2. PageIndex uses an LLM to analyze the document's structure.
3. Builds a hierarchical **tree index**, similar to a smart Table of Contents.
4. Returns a `doc_id` for future operations.

### Why No Chunking?

Instead of splitting the document into arbitrary 500-token chunks, PageIndex respects the document's natural section boundaries — chapters, sub-sections, and paragraphs — as intended by the author.


In [7]:
PDF_PATH = "../data/sample_document.pdf"  

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ../data/sample_document.pdf
✅ Uploaded!
📋 Document ID: pi-cmv50g7k800040dp4j8f8rwpx
   (Save this ID — you'll use it throughout the notebook)


In [8]:
print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!



## 🔍 Section 3: Inspect the Tree Structure

### What the Tree Looks Like

```text
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

### Each Node Has

- `node_id` — Unique ID used during retrieval.
- `title` — Section heading.
- `page_index` — Page number in the original PDF.
- `text` — Section summary (when `node_summary=True`).
- `nodes` — Child sections nested under the current node.

**Key takeaway:** This hierarchical tree structure is what the LLM reasons over during retrieval to identify the most relevant sections of the document.


In [9]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 11

🌲 Raw tree (first node):
{
  "title": "Preface",
  "node_id": "0000",
  "page_index": 1,
  "start_index": 1,
  "end_index": 15,
  "summary": "This section serves as the book's preface, containing the title page, publication and copyright details, licensing terms, publisher disclaimers, and the author's dedication. It also provides the complete table of contents, which outlines the introductory material, core chapters on artificial intelligence, machine learning, deep learning architectures (CNNs, RNNs, LSTMs, GANs), natural language processing, and reinforcement learning, as well as the appendices on Keras, TensorFlow 2, Pandas, Jupyter, and the index.",
  "text": "",
  "nodes": [
    {
      "title": "Title Page, Copyright, and Publisher Disclaimer",
      "node_id": "0001",
      "page_index": 1,
      "start_index": 1,
      "end_index": 5,
      "summary": "This section covers the title page, publication details, copyright notices, licensing terms, and dis

In [10]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Preface  (p.1)
  └─ [0001] Title Page, Copyright, and Publisher Disclaimer  (p.1)
  └─ [0002] Copyright, Dedication, and Table of Contents  (p.5)
  └─ [0003] Extended Table of Contents: Chapters 2 to 6  (p.9)
  └─ [0004] Table of Contents: Chapter 6, Appendices, and Index  (p.12)
[0005] PREFACE: THE ML AND DL LANDSCAPE  (p.16)
[0006] INTRODUCTION TO AI  (p.20)
  └─ [0007] INTRODUCTION TO AI (intro)  (p.20)
  └─ [0008] What Is Artificial Intelligence?  (p.21)
  └─ [0009] The Turing Test  (p.24)
  └─ [0010] Heuristics  (p.25)
  └─ [0011] Knowledge Representation  (p.27)
  └─ [0012] AI and Games  (p.29)
  └─ [0013] Expert Systems  (p.31)
  └─ [0014] Neural Computing  (p.32)
  └─ [0015] Evolutionary Computation  (p.33)
  └─ [0016] Natural Language Processing  (p.33)
  └─ [0017] Bioinformatics  (p.36)
  └─ [0018] Major Parts of AI  (p.37)
  └─ [0019] Code Samples  (p.40)
  └─ [0020] Summary  (p.41)
[0021] INTRODUCTION TO MACHINE LEARNING  (p.42)
  └─ [0022

In [11]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 187
   Each node = one retrievable section of the document



## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from Vector RAG.**

### Vector RAG Retrieval

```text
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```

*Problem:* Finds what's similar, not necessarily what's relevant.

### PageIndex Retrieval

```text
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```

*Advantage:* The LLM understands document structure, context, and intent to identify the most relevant sections.

### Key Takeaway

**The LLM acts like a human expert scanning a Table of Contents** to locate the sections most likely to contain the answer.


In [12]:
def llm_tree_search(query: str, tree: list) -> dict:
    """
    Core PageIndex retrieval using OpenRouter.

    Sends the query and compressed document tree to an LLM.
    Returns relevant node IDs in JSON format.
    """

    # Compress the tree to save tokens
    def compress(nodes):
        out = []

        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }

            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])

            out.append(entry)

        return out

    compressed_tree = compress(tree)

    prompt = f"""
You are given a query and a document's tree structure,
similar to a Table of Contents.

Identify the node IDs most likely to contain the answer.
Use the section titles, summaries, and hierarchy to guide
your selection.

Query:
{query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this JSON format:
{{
    "thinking": "Brief explanation of the selected sections",
    "node_list": ["node_id1", "node_id2"]
}}
"""

    # Call the model through OpenRouter
    response = llm.invoke(prompt)

    # Parse the JSON returned by the model
    content = response.content

    if isinstance(content, list):
        content = "".join(
            item["text"] for item in content
            if isinstance(item, dict) and "text" in item
        )

    return json.loads(content)

In [14]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What Are Word Embeddings?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What Are Word Embeddings?

🧠 LLM Reasoning:
The query exactly matches the section title 'What Are Word Embeddings?' at node 0102, whose summary directly defines embeddings. This section is the most likely location for the answer.

🎯 Selected Node IDs: ['0102']
